In [2]:
# as easy as loading using prior localization code

from one.api import ONE
from pathlib import Path
import yaml
import os
import wfield
import numpy as np
import pandas as pd
from prior_localization.prepare_data import prepare_widefield, prepare_pupil
from brainbox.io.one import SessionLoader
from brainwidemap.bwm_loading import load_trials_and_mask
from prior_localization.functions.utils import compute_mask
from ibl_info.prepare_data_pid import get_new_cinc_intervals
import seaborn as sns
from matplotlib import pyplot as plt
import pickle as pkl
from tqdm import tqdm
from glob import glob
import re
import concurrent.futures
import pickle as pkl
import time
from one.api import ONE
import pandas as pd
from tqdm import tqdm
from brainwidemap import bwm_query, load_good_units, load_trials_and_mask, bwm_units
from brainbox.singlecell import bin_spikes2D
from iblatlas.regions import BrainRegions
import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from ibl_info.utils import check_config, compute_animal_stats
from scipy.ndimage import convolve1d
import traceback
from scipy.stats import zscore

config = check_config()
MY_REGIONS = config["stim_prior_regions"]

In [3]:
one = ONE(
    base_url="https://openalyx.internationalbrainlab.org",
    password="international",
    silent=True,
    username="intbrainlab",
)
print("Querying BWM Units...")

units_df = bwm_units(one)
relevant_pids = units_df[units_df["Beryl"].isin(MY_REGIONS)]["pid"].unique()

bwm_df = bwm_query(one)
subset_df = bwm_df[bwm_df["pid"].isin(relevant_pids)]

Querying BWM Units...
Loading bwm_query results from fixtures/2023_12_bwm_release.csv
d16d0b38d392b18c0ce8b615ec89d60d7c901df2eeb3432986b62130af28ef01
Loading bwm_query results from fixtures/2023_12_bwm_release.csv


In [5]:
list_of_eids = subset_df["eid"].unique()

In [8]:
sl = SessionLoader(one, eid=list_of_eids[0])

In [9]:
sl.load_pupil()

/Users/dkundu/mamba/envs/info-decom/lib/python3.10/site-packages/one/util.py:428: ALFWarning: Multiple revisions: "2025-06-18", "2023-04-20"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)
(S3) /Users/dkundu/Downloads/ONE/openalyx.internationalbrainlab.org/angelakilab/Subjects/NYU-11/2020-02-18/001/alf/#2025-06-18#/_ibl_leftCamera.features.pqt: 100%|██████████| 5.33M/5.33M [00:01<00:00, 5.19MB/s]
(S3) /Users/dkundu/Downloads/ONE/openalyx.internationalbrainlab.org/angelakilab/Subjects/NYU-11/2020-02-18/001/alf/#2023-04-20#/_ibl_leftCamera.times.npy: 100%|██████████| 2.46M/2.46M [00:00<00:00, 6.36MB/s]


In [10]:
sl.load_trials()

/Users/dkundu/mamba/envs/info-decom/lib/python3.10/site-packages/one/util.py:428: ALFWarning: Multiple revisions: "", "2025-03-03"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)
(S3) /Users/dkundu/Downloads/ONE/openalyx.internationalbrainlab.org/angelakilab/Subjects/NYU-11/2020-02-18/001/alf/#2025-03-03#/_ibl_trials.stimOffTrigger_times.npy: 100%|██████████| 4.65k/4.65k [00:00<00:00, 18.7kB/s]
(S3) /Users/dkundu/Downloads/ONE/openalyx.internationalbrainlab.org/angelakilab/Subjects/NYU-11/2020-02-18/001/alf/#2025-03-03#/_ibl_trials.stimOff_times.npy: 100%|██████████| 4.65k/4.65k [00:00<00:00, 18.6kB/s]
(S3) /Users/dkundu/Downloads/ONE/openalyx.internationalbrainlab.org/angelakilab/Subjects/NYU-11/2020-02-18/001/alf/#2025-03-03#/_ibl_trials.stimOnTrigger_times.npy: 100%|██████████| 4.65k/4.65k [00:00<00:00, 17.3kB/s]
(S3) /Users/dkundu/Downloads/ONE/openalyx.internationalbrainlab.org/angelakilab/Subjects/NYU-11/2020-02-18/001/alf/#2025-03-03#/_ibl_trials.table.pqt:

In [11]:
sl.pupil

,times,pupilDiameter_raw,pupilDiameter_smooth
0,268.285319,14.503242,13.980196
1,268.301927,14.503242,13.978801
2,268.318631,14.503242,13.977874
3,268.335239,14.427560,13.977456
4,268.351943,14.384402,13.971926
...,...,...,...
307370,5386.201016,6.801762,NaN
307371,5386.217683,7.109304,NaN
307372,5386.234349,8.147023,NaN
307373,5386.251016,8.147023,NaN


In [13]:
sl.trials.stimOn_times

0       268.974705
1       317.356780
2       381.061090
3       389.144322
4       408.391916
          ...     
560    3601.904657
561    3610.636659
562    3614.053600
563    3617.720623
564    3621.174620
Name: stimOn_times, Length: 565, dtype: float64

In [14]:
from scipy.stats import zscore


def extract_baseline_pupil(trials_df, pupil_df, window=(-0.6, -0.1)):
    """
    Extracts the average pupil diameter for a specific window relative to Stimulus Onset.

    Parameters:
    - trials_df: DataFrame with 'stimOn_times'
    - pupil_df: DataFrame with 'times' and 'pupil_diameter' (or 'smooth')
    - window: tuple (start_offset, end_offset) e.g., (-0.6, -0.1)

    Returns:
    - trials_df with a new column 'pupil_baseline' (Z-scored per session is recommended later)
    """

    if not pupil_df["times"].is_monotonic_increasing:
        pupil_df = pupil_df.sort_values("times")

    p_times = pupil_df["times"].values
    p_vals = pupil_df["pupilDiameter_smooth"].values  # Or 'pupil_smooth'

    stim_times = trials_df["stimOn_times"].values
    starts = stim_times + window[0]
    ends = stim_times + window[1]

    idx_start = np.searchsorted(p_times, starts)
    idx_end = np.searchsorted(p_times, ends)

    means = np.full(len(trials_df), np.nan)  # Initialize with NaNs

    temp_means = []
    for i, j in zip(idx_start, idx_end):
        if i < j:
            chunk = p_vals[i:j]
            temp_means.append(np.nanmean(chunk))
        else:
            temp_means.append(np.nan)
    # 5. Add to DataFrame
    trials_df = trials_df.copy()
    trials_df["pupil_baseline_raw"] = temp_means

    return trials_df


def process_pupil_data(trials, pupil):
    """
    Wrapper to handle the Z-Scoring logic correctly.
    Crucial: Z-Score must be done PER MOUSE/SESSION, not globally.
    """
    trials = extract_baseline_pupil(trials, pupil)

    vals = trials["pupil_baseline_raw"].values

    mean_val = np.nanmean(vals)
    std_val = np.nanstd(vals)

    trials["pupil_z"] = (vals - mean_val) / std_val

    return trials

In [16]:
tx, mask = load_trials_and_mask(one, list_of_eids[0])

In [ ]:
tx = tx[mask]

In [18]:
tx_with_pupilometry = process_pupil_data(tx, sl.pupil)

In [33]:
tx.columns

Index(['goCueTrigger_times', 'intervals_bpod_0', 'intervals_bpod_1',
       'quiescencePeriod', 'stimOff_times', 'firstMovement_times',
       'goCue_times', 'probabilityLeft', 'response_times', 'feedbackType',
       'rewardVolume', 'contrastRight', 'choice', 'feedback_times',
       'stimOn_times', 'contrastLeft', 'intervals_0', 'intervals_1'],
      dtype='object')

In [ ]:
keep_columns = ['feedbackType', 'probabilityLeft', 'contrastRight', 'contrastLeft']